In [ ]:
try:
    from databricks.sdk import WorkspaceClient
except ImportError:
    !pip install databricks-sdk
    from databricks.sdk import WorkspaceClient

In [ ]:
# Importing Libraries
import time
import json
import requests
import pandas as pd
from databricks.sdk import WorkspaceClient
from google.colab import userdata
from concurrent.futures import ThreadPoolExecutor, as_completed

# Defining functions

In [ ]:
# ----- # ----- # ----- Get Jobs for Netflix ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_netflix(start=0, num=10):
    response = ''
    try:
        url = 'https://explore.jobs.netflix.net/api/apply/v2/jobs'
        params = {
            'domain': 'netflix.com',
            'start': start,
            'num': num
        }
        response = requests.get(url, params=params)
        response_json = json.loads(response.text)
        if 'positions' in response_json:
          return True, response_json['positions']
        else:
          return False, {'Error': response.text}
    except Exception as e:
        # print(str(e))
        # print(f"Unable to connect to Netflix: {response.text}")
        return False, {'Error': response.text}


# ----- # ----- # ----- Get Job details for Netflix ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def jobs_details_netflix(record=''):
    response = ''
    try:
        if record['index']%50 == 0:
          time.sleep(60)
          print(record['index'])
        url = f'https://explore.jobs.netflix.net/api/apply/v2/jobs/{record['id']}'
        params = {
            'domain': 'netflix.com',
            'pcs_jobs_card_index': 1,
            'pid': record['id']
        }

        response = requests.get(url, params=params)
        response_json = json.loads(response.text)
        if 'id' in response_json:
            return response_json
        else:
            return {'Error': response.text}
    except Exception as e:
        print(str(e))
        print(f"Unable to connect to Netflix: {response.text}")
        return {'Error': response.text}


# ----- # ----- # ----- Fetch Job List ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_jobs():
    limit = 10
    offset = 0
    job_df_list = []

    # Perform Initial API Call
    job_bool, job_json = jobs_netflix(offset, limit)

    # Start the loop
    while len(job_json) != 0:
        if offset%100 == 0:
          print(f'Limit: {limit}, Offset: {offset}, Count:{limit + offset}')
        job_bool, job_json = jobs_netflix(offset, limit)
        job_df_list.extend(job_json)
        offset += limit

    # Concatenate the dataset and clean
    jobs = pd.DataFrame(job_df_list)
    jobs.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/netflix_jobs.parquet', index=False)
    # jobs = jobs[['Id', 'Title', 'PostedDate', 'JobFamily', 'JobFunction', 'ShortDescriptionStr', 'PrimaryLocation', 'secondaryLocations']]
    # jobs.reset_index(inplace=True, drop=False)
    print(f'Total Jobs: {len(jobs)}')
    return jobs


# ----- # ----- # ----- Fetch Job Details ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def fetch_job_details(job_df):
    results = []

    # Parallel processing
    with ThreadPoolExecutor(max_workers=10) as executor:  # adjust workers based on API rate limits
        future_to_record = {executor.submit(jobs_details_netflix, row): row for _, row in job_df.iterrows()}
        for future in as_completed(future_to_record):
            results.append(future.result())
    results_df = pd.DataFrame(results)
    # results_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/netflix_jobs_details.parquet', index=False)
    return results_df


In [ ]:
# ----- # ----- # ----- Upload file to Databricks ----- # ----- # ----- #
# ----- # ----- # ----- # ----- # ----- # ----- # ----- # ----- #
def upload_data(data_file):
  w = WorkspaceClient(
      host = userdata.get('DATABRICKS_HOST'),
      token = userdata.get('DATABRICKS_TOKEN')
  )

  volume_path = "/Volumes/workspace/jobs_raw/job_files/" + data_file.split('/')[-1]
  with open(data_file, "rb") as f:
      w.files.upload(volume_path, f)

  print(f"Successfully uploaded {data_file} to {volume_path}")

# New Architecture

In [ ]:
job_frame = fetch_jobs()
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/netflix_jobs.parquet')
job_frame.reset_index(inplace=True, drop=False)

In [ ]:
job_details_frame = fetch_job_details(job_frame)
upload_data('/content/drive/MyDrive/Colab Notebooks/JobAI/jobs/netflix_jobs_details.parquet')